pandas is imported to load, clean, and analyze tabular data.
glob helps find all CSV files in a folder using patterns like *.csv.
os is used to work with file paths and extract filenames during data processing

In [ ]:
import pandas as pd
import glob
import os

This code searches the dataset folder and collects all CSV files using a wildcard pattern.
glob.glob() returns a list of matching file paths, helping us automatically detect all patient and demographic files.
Printing the first five entries confirms that the files were found correctly.

If you want, I can also write short explanations for the next code blocks

In [ ]:
path = "HUPA-UC Diabetes Dataset/*.csv"
import glob

files = glob.glob(path)
print(files[:5])

This code filters the list of CSV files and keeps only those whose filenames start with “HUPA”, which represent individual patient monitoring files.
os.path.basename() extracts just the filename, making the filtering accurate.
Finally, it prints how many patient files were detected in the dataset

In [ ]:
patient_files = [f for f in files if os.path.basename(f).startswith("HUPA")] # just taking files starting with HUPA
print("Patient monitoring files:", len(patient_files))

This loop reads each patient CSV file, extracts the patient ID from the filename, and adds it as a new column in the DataFrame.
Each processed DataFrame is stored in a list, and finally all of them are combined into one master dataset using pd.concat().
This creates a single unified DataFrame containing data from all patients

In [ ]:
all_data = []

for file in patient_files:
    df = pd.read_csv(file, sep=';')
    
    patient_id = os.path.basename(file).replace(".csv","")
    df["Patient_ID"] = patient_id
    
    all_data.append(df)

master_df = pd.concat(all_data, ignore_index=True)

This command displays the first five rows of the combined master dataset.
It helps verify that all patient files were loaded correctly and that the Patient_ID column was added as expected.
Using .head() is a quick way to inspect the structure and quality of the merged data

In [ ]:

master_df.head()

This command prints all column names in the master dataset.
It helps verify that every patient file was loaded correctly and that the Patient_ID column was added.
It’s also useful for checking if any columns need renaming or cleaning before analysis

In [ ]:
print(master_df.columns)

This code loads the demographics and sleep dataset into a DataFrame using pd.read_csv().
meta.head() shows the first few rows to verify the file loaded correctly, and meta.columns prints all column names so you can inspect the structure of the demographics file.

In [ ]:
meta = pd.read_csv("HUPA-UC Diabetes Dataset/T1DM_patient_sleep_demographics_with_race.csv") # reading sleep demographics.csv 
print(meta.head())
print(meta.columns)

This merges the master patient time‑series dataset with the demographics file using Patient_ID as the common key.
A left join ensures every patient’s monitoring data is kept, and matching demographic fields are added where available.
This creates one combined dataset containing both sensor data and patient attributes.

In [ ]:
master_df = master_df.merge(meta, on="Patient_ID", how="left") # merging sleep demographics with the mater data frame

In [ ]:
master_df.head()

This loop reads every CSV file, adds a patient_id column based on the filename, and stores each DataFrame in a list.
All individual DataFrames are then combined into one large dataset using pd.concat().
master.head() shows the first few rows to confirm the merge worked correctly.

In [ ]:
dfs = []

for f in files:
    if f.endswith(".csv"):
        temp = pd.read_csv(os.path.join(data_path, f), sep=";")
        temp["patient_id"] = f.replace(".csv", "")
        dfs.append(temp)

master = pd.concat(dfs, ignore_index=True)
master.head()


In [ ]:
master.columns = (
    master.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

master["time"] = pd.to_datetime(master["time"])
master = master.sort_values(["patient_id", "time"])
master = master.fillna(0)


This line saves the final cleaned master dataset as a CSV file in the Outputs folder.
Using index=False prevents pandas from adding an extra index column to the file.
It creates a ready‑to‑use dataset for analysis, modeling, or sharing with your team

In [ ]:
master.to_csv("Outputs/master_clean.csv", index=False)
